In [0]:
%sh
# -------------------------------------------------------------------------------------------
# NOTEBOOK: 01b_raw_data_preparation
# ROLE: Unzip and stage raw data files (idempotent, resource-optimized)
# -------------------------------------------------------------------------------------------

VOLUME_PATH="/Volumes/hc_regulatory_sandbox/bronze_ingestion/landing_zone"

echo "Preparing raw data files..."
echo ""

# 1. Unzip FDA Master Hub (Drugs@FDA regulatory data)
echo "[1/4] Extracting FDA Master Hub..."
unzip -o -q $VOLUME_PATH/raw_uploads/dafdata20260410.zip -d $VOLUME_PATH/fda_master_hub/
if [ $? -eq 0 ]; then
  echo "  ✓ dafdata20260410.zip extracted"
else
  echo "  ✗ FAILED to extract dafdata20260410.zip"
fi

# 2. Unzip DailyMed SPL (Structured Product Labeling data)
echo "[2/4] Extracting DailyMed SPL data (outer archive)..."
unzip -o -q $VOLUME_PATH/raw_uploads/dm_spl_daily_update_04102026.zip -d $VOLUME_PATH/spl_labels/
if [ $? -eq 0 ]; then
  echo "  ✓ dm_spl_daily_update_04102026.zip extracted to spl_labels/"
else
  echo "  ✗ FAILED to extract dm_spl_daily_update_04102026.zip"
fi

# 3. Extract nested ZIP files within spl_labels subdirectories
echo "[3/4] Extracting nested ZIP files in spl_labels subdirectories..."
echo "     This may take a few minutes for large datasets..."

zip_count=0
success_count=0
fail_count=0

# Find all ZIP files in spl_labels subdirectories and extract them
for subdir in $VOLUME_PATH/spl_labels/*/; do
  if [ -d "$subdir" ]; then
    subdir_name=$(basename "$subdir")
    echo "     Processing: $subdir_name/"
    
    # Count and extract all ZIPs in this subdirectory
    for zipfile in "$subdir"*.zip; do
      if [ -f "$zipfile" ]; then
        zip_count=$((zip_count + 1))
        
        # Extract each ZIP into the same directory, then remove the ZIP
        unzip -o -q "$zipfile" -d "$subdir" 2>/dev/null
        if [ $? -eq 0 ]; then
          success_count=$((success_count + 1))
          rm "$zipfile"  # Remove ZIP after successful extraction
        else
          fail_count=$((fail_count + 1))
        fi
      fi
    done
  fi
done

if [ $zip_count -gt 0 ]; then
  echo "  ✓ Extracted $success_count of $zip_count nested ZIP files"
  if [ $fail_count -gt 0 ]; then
    echo "  ⚠️  $fail_count files failed to extract"
  fi
else
  echo "  ℹ️  No nested ZIP files found (may have been previously extracted)"
fi

# 4. Copy DAC National Downloadable File (Provider Registry)
echo "[4/4] Copying DAC National Downloadable File..."
if [ -f "$VOLUME_PATH/raw_uploads/DAC_NationalDownloadableFile.csv" ]; then
  cp $VOLUME_PATH/raw_uploads/DAC_NationalDownloadableFile.csv $VOLUME_PATH/provider_registry/
  if [ $? -eq 0 ]; then
    echo "  ✓ DAC_NationalDownloadableFile.csv copied"
  else
    echo "  ✗ FAILED to copy DAC_NationalDownloadableFile.csv"
  fi
else
  echo "  ⚠️  SKIPPED - DAC_NationalDownloadableFile.csv not found in raw_uploads/"
  echo "     Upload this file to: $VOLUME_PATH/raw_uploads/"
fi

echo ""
echo "============================================================"
echo "✅ Raw data preparation complete"
echo "============================================================"
echo "Extracted to:"
echo "  • $VOLUME_PATH/fda_master_hub/"
echo "  • $VOLUME_PATH/spl_labels/ (with subdirectories)"
echo "  • $VOLUME_PATH/provider_registry/"
echo "============================================================"
echo ""
echo "Next steps:"
echo "1. Run 02_ingest_bronze_raw to load data into Delta tables"
echo "2. Note: SPL data is organized in subdirectories (prescription, otc, etc.)"
echo "============================================================"